In [1]:
import logging
import os.path as osp
import copy
import time
from functools import partial
from typing import Optional

import numpy as np
import torch
from torch.nn import functional as F
import torch_geometric.transforms as T
from numpy.random import default_rng
from ogb.graphproppred import PygGraphPropPredDataset
from torch_geometric.datasets import (
    Actor,
    GNNBenchmarkDataset,
    Planetoid,
    TUDataset,
    WebKB,
    WikipediaNetwork,
    ZINC,
)
from torch_geometric.graphgym.config import cfg
from torch_geometric.graphgym.loader import load_pyg, load_ogb, set_dataset_attr
from torch_geometric.graphgym.register import register_loader

from lrgb_exps.graphgps.loader.dataset.aqsol_molecules import AQSOL
from lrgb_exps.graphgps.loader.dataset.coco_superpixels import COCOSuperpixels
from lrgb_exps.graphgps.loader.dataset.malnet_tiny import MalNetTiny
from lrgb_exps.graphgps.loader.dataset.voc_superpixels import VOCSuperpixels
from lrgb_exps.graphgps.loader.split_generator import (
    prepare_splits,
    set_dataset_splits,
)
from lrgb_exps.graphgps.transform.posenc_stats import compute_posenc_stats
from lrgb_exps.graphgps.transform.task_preprocessing import (
    task_specific_preprocessing,
)
from lrgb_exps.graphgps.transform.transforms import (
    pre_transform_in_memory,
    typecast_x,
    concat_x_and_pos,
    clip_graphs_to_size,
)
from torch_geometric.data import Data
from longrange.task_preprocessing import add_distance_attrs
from ogb.utils.features import get_atom_feature_dims
from torch_geometric.transforms import BaseTransform

        
from lrgb_exps.graphgps.loader.dataset.peptides_structural import (
            PeptidesStructuralDataset,
        )

In [2]:
dataset = PeptidesStructuralDataset(
            root='lrgb_exps/datasets/peptides_issues',
            transform=None,
            pre_transform=None,
            subset=-1,
        )

In [3]:
from torch_geometric.utils import to_networkx
import networkx as nx




count_disc = 0 
count_conn = 0 
for i in range(len(dataset)):
    G = to_networkx(dataset[i], to_undirected=True)
    num_components = nx.number_connected_components(G)
    is_disconnected = num_components > 1

    if is_disconnected:
        count_disc += 1
        print("num_components: ", num_components)
        print("Disconnected graph size: ", dataset[i].num_nodes)
    else:
        count_conn += 1
print("Disconnected graphs: ", count_disc)
print("Connected graphs: ", count_conn)

s_dict = dataset.get_idx_split()
dataset.split_idxs = [s_dict[s] for s in ["train", "val", "test"]]

print("Train: ", len(dataset.split_idxs[0]))
count_disc = 0 
count_conn = 0 
for i in dataset.split_idxs[0]:
    if dataset[i].has_isolated_nodes():
        count_disc += 1
    else:
        count_conn += 1
print("Disconnected graphs: ", count_disc)
print("Connected graphs: ", count_conn)


num_components:  2
Disconnected graph size:  215
num_components:  6
Disconnected graph size:  185
num_components:  7
Disconnected graph size:  161
num_components:  2
Disconnected graph size:  165
num_components:  6
Disconnected graph size:  133
num_components:  2
Disconnected graph size:  186
num_components:  7
Disconnected graph size:  151
num_components:  2
Disconnected graph size:  215
num_components:  2
Disconnected graph size:  148
num_components:  6
Disconnected graph size:  80
num_components:  2
Disconnected graph size:  64
num_components:  2
Disconnected graph size:  139
num_components:  6
Disconnected graph size:  107
num_components:  2
Disconnected graph size:  67
num_components:  2
Disconnected graph size:  64
num_components:  6
Disconnected graph size:  46
num_components:  2
Disconnected graph size:  196
num_components:  7
Disconnected graph size:  154
num_components:  6
Disconnected graph size:  143
num_components:  5
Disconnected graph size:  10
num_components:  2
Disconn

In [ ]:
def remove_hydrogens(data: Data, atomic_number_index: int = 0) -> Data:
    # Step 1: Identify non-hydrogen atoms
    print("data.x: ", data.x)
    is_not_hydrogen = data.x[:, atomic_number_index] != 1
    keep_idx = is_not_hydrogen.nonzero(as_tuple=True)[0]
    
    # Step 2: Remap node indices (since we'll shrink x)
    old_to_new = -torch.ones(data.num_nodes, dtype=torch.long)
    old_to_new[keep_idx] = torch.arange(keep_idx.size(0))

    # Step 3: Filter x (node features)
    data.x = data.x[keep_idx]

    # Step 4: Filter edge_index to only include edges between non-H atoms
    mask = is_not_hydrogen[data.edge_index[0]] & is_not_hydrogen[data.edge_index[1]]
    data.edge_index = data.edge_index[:, mask]
    data.edge_index = old_to_new[data.edge_index]

    # Optional: Filter any other node-wise attributes (like pos, y, etc.)
    if hasattr(data, 'pos'):
        data.pos = data.pos[keep_idx]
    if hasattr(data, 'y') and data.y.size(0) == data.num_nodes:
        data.y = data.y[keep_idx]

    return data

def find_isolated_nodes(data: Data):
    num_nodes = data.num_nodes
    all_nodes = torch.arange(num_nodes)

    # Remove self-loops (edges where source == target)
    src, dst = data.edge_index
    non_self_loop_mask = src != dst
    filtered_src = src[non_self_loop_mask]
    filtered_dst = dst[non_self_loop_mask]

    # Nodes that are involved in any *non-self-loop* edges
    connected_nodes = torch.unique(torch.cat([filtered_src, filtered_dst]))

    # Isolated nodes are not involved in any of those
    isolated_nodes = all_nodes[~torch.isin(all_nodes, connected_nodes)]

    return isolated_nodes

for i in range(len(dataset)):
    nodes = find_isolated_nodes(dataset[i])
    if len(nodes) > 0:
        print("Num Isolated nodes: ", len(nodes))
        print(dataset[i].x[nodes])


tensor([[5, 0, 4, 5, 4, 0, 2, 0, 0],
        [5, 0, 4, 5, 4, 0, 2, 0, 0],
        [5, 0, 4, 5, 4, 0, 2, 0, 0]])
tensor([[5, 0, 4, 5, 4, 0, 2, 0, 0],
        [5, 0, 4, 5, 4, 0, 2, 0, 0],
        [5, 0, 4, 5, 4, 0, 2, 0, 0],
        [5, 0, 4, 5, 4, 0, 2, 0, 0],
        [5, 0, 4, 5, 4, 0, 2, 0, 0]])
tensor([[5, 0, 4, 5, 4, 0, 2, 0, 0]])
tensor([[5, 0, 4, 5, 4, 0, 2, 0, 0],
        [5, 0, 4, 5, 4, 0, 2, 0, 0],
        [5, 0, 4, 5, 4, 0, 2, 0, 0],
        [5, 0, 4, 5, 4, 0, 2, 0, 0]])
tensor([[5, 0, 4, 5, 4, 0, 2, 0, 0],
        [5, 0, 4, 5, 4, 0, 2, 0, 0],
        [5, 0, 4, 5, 4, 0, 2, 0, 0],
        [5, 0, 4, 5, 4, 0, 2, 0, 0],
        [5, 0, 4, 5, 4, 0, 2, 0, 0]])
tensor([[5, 0, 4, 5, 4, 0, 2, 0, 0]])
tensor([[5, 0, 4, 5, 4, 0, 2, 0, 0],
        [5, 0, 4, 5, 4, 0, 2, 0, 0],
        [5, 0, 4, 5, 4, 0, 2, 0, 0],
        [5, 0, 4, 5, 4, 0, 2, 0, 0]])
tensor([[5, 0, 4, 5, 4, 0, 2, 0, 0]])
tensor([[5, 0, 4, 5, 4, 0, 2, 0, 0],
        [5, 0, 4, 5, 4, 0, 2, 0, 0],
        [5, 0, 4, 5, 4, 0, 2, 